# Grokking Arithmetic: Tiny Transformer for +, −, ×, ÷

**Goal:** Train the smallest possible autoregressive transformer that learns arithmetic operations.

**Inspiration:** The famous 777-parameter transformer that learned 10-digit addition with 99% accuracy through "grokking" — a sudden jump from memorization to generalization.

**Target:**
- <1,000 parameters
- >99% accuracy on held-out test set
- All four operations: +, −, ×, ÷

**Key insight:** The model can't memorize (would need astronomical parameters), so if it succeeds, it must have learned the actual algorithm.

## 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm
import random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Data Generation

We'll work with modular arithmetic (mod p) to keep numbers bounded.
Format: `a OP b = c` where OP ∈ {+, -, *, /}

Using mod 97 (prime) so division is always defined.

In [ ]:
class ArithmeticDataset(Dataset):
    """
    Dataset for modular arithmetic.
    Format: a OP b = c (all mod p)
    
    Vocabulary:
    - 0-96: digits (mod 97)
    - 97: '+'
    - 98: '-'
    - 99: '*'
    - 100: '/'
    - 101: '='
    - 102: <PAD>
    - 103: <EOS>
    """
    
    def __init__(self, operation='add', p=97, train=True, train_fraction=0.5, seed=42):
        self.p = p
        self.operation = operation
        
        # Vocabulary
        self.vocab_size = p + 7  # digits + ops + special
        self.op_tokens = {'+': p, '-': p+1, '*': p+2, '/': p+3}
        self.eq_token = p + 4
        self.pad_token = p + 5
        self.eos_token = p + 6
        
        # Generate all (a, b) pairs
        all_pairs = [(a, b) for a in range(p) for b in range(p)]
        
        # For division, b != 0
        if operation == 'div':
            all_pairs = [(a, b) for a, b in all_pairs if b != 0]
        
        # Split train/test
        random.seed(seed)
        random.shuffle(all_pairs)
        split = int(len(all_pairs) * train_fraction)
        
        self.pairs = all_pairs[:split] if train else all_pairs[split:]
        
    def __len__(self):
        return len(self.pairs)
    
    def compute(self, a, b):
        """Compute a OP b mod p"""
        if self.operation == 'add':
            return (a + b) % self.p
        elif self.operation == 'sub':
            return (a - b) % self.p
        elif self.operation == 'mul':
            return (a * b) % self.p
        elif self.operation == 'div':
            # Modular inverse using Fermat's little theorem
            b_inv = pow(b, self.p - 2, self.p)
            return (a * b_inv) % self.p
    
    def __getitem__(self, idx):
        a, b = self.pairs[idx]
        c = self.compute(a, b)
        
        op_token = self.op_tokens[{'+': '+', 'add': '+', '-': '-', 'sub': '-', 
                                    '*': '*', 'mul': '*', '/': '/', 'div': '/'}[self.operation]]
        
        # Sequence: a, OP, b, =, c, <EOS>
        # Input:  a, OP, b, =
        # Target: OP, b, =, c (shifted)
        
        input_seq = torch.tensor([a, op_token, b, self.eq_token], dtype=torch.long)
        target = torch.tensor(c, dtype=torch.long)  # Just predict the answer
        
        return input_seq, target

# Test dataset
ds = ArithmeticDataset('add', p=97, train=True)
print(f"Train size: {len(ds)}")
print(f"Vocab size: {ds.vocab_size}")
x, y = ds[0]
print(f"Sample: {x.tolist()} -> {y.item()}")

## 3. Tiny Transformer Architecture

**Design for minimal parameters:**
- Tiny embedding dimension (16-32)
- Single attention layer
- 1-2 heads
- No MLP or minimal MLP
- Shared input/output embeddings

In [ ]:
class TinyArithmeticTransformer(nn.Module):
    """
    Minimal autoregressive transformer for arithmetic.
    
    Target: <1000 parameters
    """
    
    def __init__(self, vocab_size=104, embed_dim=16, num_heads=1, max_len=5):
        super().__init__()
        
        self.embed_dim = embed_dim
        self.vocab_size = vocab_size
        
        # Token embeddings
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        
        # Learnable position embeddings
        self.pos_embed = nn.Parameter(torch.zeros(1, max_len, embed_dim))
        
        # Single attention layer
        self.attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )
        self.norm = nn.LayerNorm(embed_dim)
        
        # Output head (tied with input embeddings)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
        # Tie weights
        self.head.weight = self.token_embed.weight
        
        self._init_weights()
        
    def _init_weights(self):
        nn.init.normal_(self.token_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed, std=0.02)
        
    def forward(self, x):
        """
        x: (batch, seq_len) token indices
        Returns: (batch, vocab_size) logits for next token
        """
        B, T = x.shape
        
        # Embed tokens + positions
        tok_emb = self.token_embed(x)  # (B, T, embed_dim)
        pos_emb = self.pos_embed[:, :T, :]
        x = tok_emb + pos_emb
        
        # Causal attention mask
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        
        # Self-attention
        x_norm = self.norm(x)
        attn_out, _ = self.attention(x_norm, x_norm, x_norm, attn_mask=mask)
        x = x + attn_out
        
        # Get last position and project to vocab
        last_hidden = x[:, -1, :]  # (B, embed_dim)
        logits = self.head(last_hidden)  # (B, vocab_size)
        
        return logits


def count_parameters(model):
    """Count trainable parameters."""
    # Note: tied weights should only be counted once
    seen = set()
    total = 0
    for name, param in model.named_parameters():
        if param.data_ptr() not in seen:
            seen.add(param.data_ptr())
            total += param.numel()
    return total


# Test model
model = TinyArithmeticTransformer(vocab_size=ds.vocab_size, embed_dim=16, num_heads=1)
total_params = count_parameters(model)

print(f"\nParameter count: {total_params:,}")
print(f"Target: <1,000")
print(f"Status: {'PASS ✅' if total_params < 1000 else 'OVER BUDGET ❌'}")

## 4. Training Loop with Grokking Detection

Grokking requires:
- High weight decay (regularization)
- Long training (well past overfitting)
- Full batch training works best

In [ ]:
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        logits = model(inputs)
        loss = F.cross_entropy(logits, targets)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * inputs.size(0)
        preds = logits.argmax(dim=-1)
        correct += (preds == targets).sum().item()
        total += inputs.size(0)
    
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        logits = model(inputs)
        loss = F.cross_entropy(logits, targets)
        
        total_loss += loss.item() * inputs.size(0)
        preds = logits.argmax(dim=-1)
        correct += (preds == targets).sum().item()
        total += inputs.size(0)
    
    return total_loss / total, correct / total

In [ ]:
# Training configuration for grokking
OPERATION = 'add'  # Start with addition
P = 97  # Prime modulus
EMBED_DIM = 7
NUM_HEADS = 1
BATCH_SIZE = 512  # Large batch
NUM_EPOCHS = 10000  # Long training for grokking!
LEARNING_RATE = 0.02
WEIGHT_DECAY = 1.0  # High weight decay is crucial!

# Create datasets
train_ds = ArithmeticDataset(OPERATION, p=P, train=True, train_fraction=0.5)
test_ds = ArithmeticDataset(OPERATION, p=P, train=False, train_fraction=0.5)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Operation: {OPERATION} (mod {P})")
print(f"Train samples: {len(train_ds)}")
print(f"Test samples: {len(test_ds)}")

# Create model
model = TinyArithmeticTransformer(
    vocab_size=train_ds.vocab_size,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print(f"\nModel parameters: {count_parameters(model):,}")

In [ ]:
# Training with grokking detection
history = {
    'train_loss': [], 'train_acc': [],
    'test_loss': [], 'test_acc': []
}

best_test_acc = 0
grokking_epoch = None

print(f"Training for {NUM_EPOCHS} epochs (watching for grokking)...")
print("="*70)

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, device)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    
    # Detect grokking (sudden jump in test accuracy)
    if test_acc > best_test_acc:
        if test_acc > 0.99 and best_test_acc < 0.5 and grokking_epoch is None:
            grokking_epoch = epoch
            print(f"\n🎯 GROKKING DETECTED at epoch {epoch}!")
        best_test_acc = test_acc
        torch.save(model.state_dict(), 'best_arithmetic_model.pth')
    
    # Log every 100 epochs or on significant events
    if (epoch + 1) % 500 == 0 or epoch == 0 or test_acc > 0.99:
        print(f"Epoch {epoch+1:5d}/{NUM_EPOCHS} | "
              f"Train: {train_acc*100:.2f}% | "
              f"Test: {test_acc*100:.2f}% | "
              f"Best: {best_test_acc*100:.2f}%")
    
    # Early stop if we've grokked
    if test_acc > 0.999:
        print(f"\n🏆 Perfect accuracy reached at epoch {epoch+1}!")
        break

print("="*70)
print(f"\n🎯 Final Test Accuracy: {test_acc*100:.2f}%")
print(f"🏆 Best Test Accuracy: {best_test_acc*100:.2f}%")
if grokking_epoch:
    print(f"⚡ Grokking occurred at epoch: {grokking_epoch}")
print(f"\n{'SUCCESS! ✅' if best_test_acc > 0.99 else 'Below 99% target ❌'}")

## 5. Visualize Grokking Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', alpha=0.7)
axes[0].plot(history['test_loss'], label='Test', alpha=0.7)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curve')
axes[0].legend()
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot([a*100 for a in history['train_acc']], label='Train', alpha=0.7)
axes[1].plot([a*100 for a in history['test_acc']], label='Test', alpha=0.7)
axes[1].axhline(y=99, color='r', linestyle='--', label='99% target', alpha=0.5)
if grokking_epoch:
    axes[1].axvline(x=grokking_epoch, color='g', linestyle=':', label=f'Grokking (epoch {grokking_epoch})')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title(f'Grokking Curve ({OPERATION} mod {P})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('grokking_curve.png', dpi=150)
plt.show()

## 6. Test the Model

In [ ]:
@torch.no_grad()
def test_arithmetic(model, a, b, operation, p, dataset):
    """Test model on a specific arithmetic problem."""
    model.eval()
    
    op_symbol = {'+': '+', 'add': '+', '-': '-', 'sub': '-', 
                 '*': '*', 'mul': '*', '/': '/', 'div': '/'}[operation]
    op_token = dataset.op_tokens[op_symbol]
    
    # Create input
    input_seq = torch.tensor([[a, op_token, b, dataset.eq_token]], device=device)
    
    # Get prediction
    logits = model(input_seq)
    pred = logits.argmax(dim=-1).item()
    
    # Compute ground truth
    true = dataset.compute(a, b)
    
    return pred, true, pred == true


# Test on some examples
print(f"Testing {OPERATION} (mod {P}):\n")
print(f"{'Problem':<20} {'Predicted':<12} {'True':<12} {'Correct'}")
print("-" * 55)

test_cases = [(17, 23), (5, 15), (1, 29), (0, 20), (28, 9)]
for a, b in test_cases:
    pred, true, correct = test_arithmetic(model, a, b, OPERATION, P, train_ds)
    op = {'+': '+', 'add': '+', '-': '-', 'sub': '-', '*': '*', 'mul': '*', '/': '/', 'div': '/'}[OPERATION]
    print(f"{a} {op} {b} (mod {P})" + " "*(12-len(str(a)+str(b))) + 
          f"{pred:<12} {true:<12} {'✅' if correct else '❌'}")

## 7. Summary

In [ ]:
print("="*60)
print("GROKKING ARITHMETIC EXPERIMENT SUMMARY")
print("="*60)
print(f"\n📊 Model: Tiny Arithmetic Transformer")
print(f"   - Parameters: {count_parameters(model):,}")
print(f"   - Embedding dim: {EMBED_DIM}")
print(f"   - Attention heads: {NUM_HEADS}")
print(f"   - Layers: 1")
print(f"\n🔢 Task: {OPERATION} (mod {P})")
print(f"   - Train samples: {len(train_ds)}")
print(f"   - Test samples: {len(test_ds)}")
print(f"\n🎯 Results:")
print(f"   - Best test accuracy: {best_test_acc*100:.2f}%")
print(f"   - Target: >99%")
print(f"   - Status: {'ACHIEVED ✅' if best_test_acc > 0.99 else 'NOT ACHIEVED ❌'}")
if grokking_epoch:
    print(f"   - Grokking epoch: {grokking_epoch}")
print(f"\n📈 Training:")
print(f"   - Epochs: {len(history['train_acc'])}")
print(f"   - Learning rate: {LEARNING_RATE}")
print(f"   - Weight decay: {WEIGHT_DECAY}")
print("="*60)